# 固定 150 题：数据集、源证据与评测器一致性 QA

本 Notebook 是 `REPORT.md` 的可执行 companion。它调用 `audit_suite.py` 的同一套全量、无题号分支规则，并把重算结果与已归档 JSON 逐项核对；不会修改数据集、基线、索引或评测运行。

## tl;dr

150/150 case-grain 连接、唯一性、manifest/run/source/index 一致性可复现；当前 easy=40/50、medium=35/50，均距目标 8 题。全套反事实重算也复现了当前 evaluator 的系统性盲区，因此新基线登记应继续 blocked。

## Context & Methods

模式：一次性固定快照的数据质量与分析 QA companion。粒度是一行一个 immutable `case_id`。Notebook 读取冻结 manifests、`live-cand-max-mono-01` 归档结果、只读 SQLite evidence 与声明 PDF，并复用审计模块重算 150 行。

### Key Assumptions

- 反事实可通过证明 evaluator 的可接受集合过宽，不证明每个生产通过答案实际错误。
- PDF 可提取文本 gap 只标 source-ambiguity candidate；未进行页面视觉签核。
- near-threshold false-negative candidate 必须由独立盲审确认。
- 当前正确率取冻结归档的 `metrics.case_outcome.final_answer_accuracy`；strict pass 是独立指标。

In [1]:
from collections import Counter
from hashlib import sha256
import json
from pathlib import Path
import subprocess
import textwrap

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
assert repo_root is not None, 'repository root containing pyproject.toml was not found'
qa_root = repo_root / 'docs/evaluation-baselines/2026-08-10-three-difficulty/remediation-2026-08-10/suite-source-evaluator-qa'
project_python = repo_root / '.venv/Scripts/python.exe'
assert project_python.is_file(), 'project virtualenv Python was not found'
print(f'repo_root={repo_root}')
print(f'qa_root={qa_root}')

repo_root=D:\Learning\agent\STT
qa_root=D:\Learning\agent\STT\docs\evaluation-baselines\2026-08-10-three-difficulty\remediation-2026-08-10\suite-source-evaluator-qa


## Data

全量重算并与归档产物比较。`generated_at` 是本次重算时间，故只排除该字段；其余 summary、150 行 cases 与 findings 必须完全一致。

In [2]:
stored_summary = json.loads((qa_root / 'summary.json').read_text(encoding='utf-8'))
stored_cases = json.loads((qa_root / 'cases.json').read_text(encoding='utf-8'))
stored_findings = json.loads((qa_root / 'findings.json').read_text(encoding='utf-8'))
verification_code = textwrap.dedent(r'''
    import json
    from pathlib import Path
    import sys

    qa_root = Path(sys.argv[1])
    sys.path.insert(0, str(qa_root))
    import audit_suite as module
    fresh = module.build_audit()
    stored_summary = json.loads((qa_root / 'summary.json').read_text(encoding='utf-8'))
    stored_cases = json.loads((qa_root / 'cases.json').read_text(encoding='utf-8'))
    stored_findings = json.loads((qa_root / 'findings.json').read_text(encoding='utf-8'))
    fresh_summary = {key: value for key, value in fresh.items() if key != 'cases'}
    without_time = lambda value: {key: item for key, item in value.items() if key != 'generated_at'}
    assert without_time(fresh_summary) == without_time(stored_summary)
    assert fresh['cases'] == stored_cases
    assert fresh['findings'] == stored_findings
    print(json.dumps({
        'full_recompute_matches_archived_artifacts': True,
        'joined_cases': fresh['input_checks']['joined_case_count'],
        'finding_count': len(fresh['findings']),
    }))
''')
completed = subprocess.run(
    [str(project_python), '-c', verification_code, str(qa_root)],
    cwd=repo_root,
    check=True,
    capture_output=True,
    text=True,
)
fresh_verification = json.loads(completed.stdout)
assert fresh_verification['full_recompute_matches_archived_artifacts'] is True
fresh_verification

{'full_recompute_matches_archived_artifacts': True,
 'joined_cases': 150,
 'finding_count': 15}

## Results

先核对 intended grain、唯一性、分层、孤儿集合、哈希和 source/index 连接，再核对报告中的目标差额与关键反事实覆盖。

In [3]:
checks = stored_summary['input_checks']
case_ids = [row['case_id'] for row in stored_cases]
difficulty_counts = Counter(row['difficulty'] for row in stored_cases)
assert len(stored_cases) == len(set(case_ids)) == 150
assert difficulty_counts == {'easy': 50, 'medium': 50, 'hard': 50}
assert checks['dataset_without_result'] == []
assert checks['result_without_dataset'] == []
assert checks['manifest_hashes_match_suite'] is True
assert checks['manifest_hashes_match_archived_run'] is True
assert checks['source_hash_matches'] is True
assert checks['source_page_count_matches'] is True
assert sum(row['source_integrity_ok'] for row in stored_cases) == 150
assert sum(row['archived_evaluator_drift'] for row in stored_cases) == 0
quality_evidence = {
    'row_count': len(stored_cases),
    'unique_case_ids': len(set(case_ids)),
    'difficulty_counts': dict(difficulty_counts),
    'dataset_orphans': len(checks['dataset_without_result']),
    'result_orphans': len(checks['result_without_dataset']),
    'source_integrity_pass': sum(row['source_integrity_ok'] for row in stored_cases),
    'archived_evaluator_drift': sum(row['archived_evaluator_drift'] for row in stored_cases),
}
quality_evidence

{'row_count': 150,
 'unique_case_ids': 150,
 'difficulty_counts': {'easy': 50, 'medium': 50, 'hard': 50},
 'dataset_orphans': 0,
 'result_orphans': 0,
 'source_integrity_pass': 150,
 'archived_evaluator_drift': 0}

In [4]:
difficulty_summary = stored_summary['difficulty_summary']
headline_rows = []
for difficulty in ('easy', 'medium', 'hard'):
    row = difficulty_summary[difficulty]
    headline_rows.append({
        'difficulty': difficulty,
        'cases': row['case_count'],
        'correct': row['production_correct'],
        'final_accuracy': row['production_final_accuracy'],
        'minimum_correct': row['target']['minimum_correct'],
        'shortfall': row['target_shortfall_cases'],
        'strict_pass': row['production_strict_pass'],
    })
assert headline_rows[0]['correct'] == 40 and headline_rows[0]['shortfall'] == 8
assert headline_rows[1]['correct'] == 35 and headline_rows[1]['shortfall'] == 8
assert headline_rows[2]['correct'] == 23 and headline_rows[2]['minimum_correct'] is None
headline_rows

[{'difficulty': 'easy',
  'cases': 50,
  'correct': 40,
  'final_accuracy': 0.8,
  'minimum_correct': 48,
  'shortfall': 8,
  'strict_pass': 35},
 {'difficulty': 'medium',
  'cases': 50,
  'correct': 35,
  'final_accuracy': 0.7,
  'minimum_correct': 43,
  'shortfall': 8,
  'strict_pass': 33},
 {'difficulty': 'hard',
  'cases': 50,
  'correct': 23,
  'final_accuracy': 0.46,
  'minimum_correct': None,
  'shortfall': None,
  'strict_pass': 23}]

In [5]:
issue_counts = stored_summary['issue_counts']
expected_issue_counts = {
    'UNSUPPORTED_FALSE_ADDITION_CAN_PASS': 145,
    'FABRICATED_FACT_WITH_REFUSAL_CAN_PASS': 5,
    'EXPECTED_TERMS_ONLY_CAN_PASS': 49,
    'MULTI_BRANCH_TRUNCATION_CAN_PASS': 28,
    'NUMERIC_MUTATION_CAN_PASS': 14,
}
for issue_code, expected_count in expected_issue_counts.items():
    case_count = sum(issue_code in row['issue_codes'] for row in stored_cases)
    assert issue_counts[issue_code] == case_count == expected_count
expected_issue_counts

{'UNSUPPORTED_FALSE_ADDITION_CAN_PASS': 145,
 'FABRICATED_FACT_WITH_REFUSAL_CAN_PASS': 5,
 'EXPECTED_TERMS_ONLY_CAN_PASS': 49,
 'MULTI_BRANCH_TRUNCATION_CAN_PASS': 28,
 'NUMERIC_MUTATION_CAN_PASS': 14}

In [6]:
checksum_rows = []
for line in (qa_root / 'SHA256SUMS').read_text(encoding='utf-8').splitlines():
    expected_digest, name = line.split('  ', 1)
    if name == 'suite_source_evaluator_qa.ipynb':
        continue  # executing the notebook legitimately changes its own bytes
    payload = (qa_root / name).read_bytes()
    actual_digest = sha256(payload).hexdigest()
    assert actual_digest == expected_digest, name
    checksum_rows.append({'artifact': name, 'sha256_ok': True})
print(f'verified_checksum_entries={len(checksum_rows)}')
checksum_rows

verified_checksum_entries=7


[{'artifact': 'REPORT.md', 'sha256_ok': True},
 {'artifact': 'artifact.json', 'sha256_ok': True},
 {'artifact': 'audit_suite.py', 'sha256_ok': True},
 {'artifact': 'cases.csv', 'sha256_ok': True},
 {'artifact': 'cases.json', 'sha256_ok': True},
 {'artifact': 'findings.json', 'sha256_ok': True},
 {'artifact': 'summary.json', 'sha256_ok': True}]

## Takeaways

- **输入完整性：ready。** 150 个 case_id 唯一且 1:1 连接，难度各 50，manifest/run/source/index 连接与归档 evaluator 重算一致。
- **当前 final-accuracy 解释：needs revision。** 145 个可回答题的虚假附加断言反例、5 个拒答夹带伪造事实反例，以及多分支/关键数值反例均可复现。
- **新基线登记：blocked。** easy 与 medium 各差 8 题，且当前二元合同还不是最终事实正确性的充分证明。
- **保留限制：** source ambiguity 尚未做页面级视觉签核；近阈值误杀候选尚待独立盲审。